In [ ]:
# ========== 导入：机票助手（Gemma Tools + Gradio） ==========

# 标准库 os：读 GEMINI_API_KEY
import os
# random：本练习主路径未必用到；原 import 保留
import random
# Gradio：快速启动 ChatInterface
import gradio as gr
# Google GenAI 原生客户端
from google import genai
# OpenAI 客户端（本练习主路径以 GenAI 为主；原 import 保留）
from openai import OpenAI
# types：Tool / FunctionDeclaration / Content / Config 等
from google.genai import types
# 从 .env 加载密钥
from dotenv import load_dotenv
# IPython 显示工具（可选）
from IPython.display import Markdown, display


In [ ]:
# ========== 环境变量：加载并粗检 GEMINI_API_KEY ==========

# override=True：.env 覆盖进程里已有同名变量
load_dotenv(override=True)
api_key = os.getenv('GEMINI_API_KEY')

# 检查钥匙：排错用英文 print 保持原样
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("AQ."):
    print("An API key was found, but it doesn't start AQ. please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


# Gemma 4 API 调用

用原生 **Google GenAI** 客户端调用 `gemma-4-31b-it`，并在后面接上 **票价查询 / 改价** 工具与 Gradio 聊天界面。


In [ ]:
# ========== 初始化 Google GenAI 客户端 ==========

# 初始化本机 Google 客户端
# 它会自动获取 GEMINI_API_KEY 环境变量
client = genai.Client()


In [ ]:
# ========== call_gemma：带 system_instruction 的简单封装 ==========

def call_gemma(s_message, message):
    """带 system_instruction 调用 Gemma（原文 docstring 多写引号导致 SyntaxError，已做最小修复）。"""
    # 1. 用户留言：Content(role=user, parts=[文本])
    messages = [
       types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]
        )
    ]

    # 2. 放置系统规则：s_message 作为 system_instruction
    config = types.GenerateContentConfig(
        system_instruction=s_message,
    )
    # 3. 使用原生客户端调用模型（model id 注释可换变体，字符串不改）
    response = client.models.generate_content(
        model='gemma-4-31b-it', # Swap out with your preferred Gemma 4 variant
        contents=messages,
        config=config
    )

    # 返回模型纯文本
    return response.text


In [ ]:
# ========== 手动试调用（默认注释掉，避免一运行就花额度） ==========

# call_gemma(system_message, '你好')


In [ ]:
# ==========（可选）旧式 Tool 声明示例：单城市 get_ticket_price ==========

# Gemma 工具的描述和函数声明
# 说明：后面 chat() 实际把 Python 函数直接放进 tools=；本格保留 Schema 写法作对照
ticket_price_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="get_ticket_price",
            description="Get the airline ticket price for a city",
            parameters={
                "type": "OBJECT",
                "properties": {
                    "destination_city": {
                        "type": "STRING",
                        "description": "Destination city name"
                    }
                },
                "required": ["destination_city"]
            }
        )
    ]
)


# 不止一座城市：查询多个城市，并可添加 / 更新票价

1. 可以一次询问多个城市的票价。
2. 可以添加或更新「城市 → 价格」。
3. LLM 只会调用 `chat` 里 `AVAILABLE_FUNCTIONS` 注册过的函数；要加新能力，先写 Python 函数再登记。
4. 本练习的 `chat` 遵循 **Google GenAI SDK** 常见写法：把函数对象直接放进 `tools=`，由 SDK 推断 schema（不一定再手写 `FunctionDeclaration`）。


In [ ]:
# ========== 价目表 + get_ticket_prices：支持多城市查询 ==========

# 内存字典：城市小写英文名 → 价格字符串（可运行数据，不翻译 key）
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_prices(destination_cities: list[str]):
    """返回所请求城市的机票价格。
    声明函数参数的类型以避免错误"""
    # 若模型偶尔传入单个字符串，也包成列表，避免 for 逐字符迭代
    if isinstance(destination_cities, str):
        destination_cities = [destination_cities]

    # 返回结构化 dict，方便再喂回模型做自然语言总结
    return {
        "prices": [
            {
                "city": city,
                # 查表；未知城市用英文占位句（影响行为的字符串不改）
                "price": ticket_prices.get(
                    city.lower(),
                    "Unknown ticket price"
                )
            }
            for city in destination_cities
        ]
    }


In [ ]:
# ========== System Prompt + set_price_ticket：创建/更新票价 ==========

# system_message：告诉模型何时用哪个工具；英文指令整段保留
system_message = """
You are a travel assistant.

Use get_ticket_prices when the user asks for ticket prices.

Use set_price_ticket when the user asks to:
- create a ticket price
- update a ticket price
- change a ticket price

Always pass city names in lowercase.
"""

def set_price_ticket(city: str, price: float):
    """创建或更新城市门票价格。"""

    # 规范化：小写 + 去首尾空白，和查询侧 .lower() 对齐
    city = city.lower().strip()

    # 记录是「更新已有」还是「新建」
    existed = city in ticket_prices

    # 存成带 $ 的字符串，和种子数据风格一致
    ticket_prices[city] = f"${price}"

    # 结构化回执：给模型和下一步回复用
    return {
        "success": True,
        "action": "updated" if existed else "created",
        "city": city,
        "price": f"${price}",
        "all_prices": ticket_prices
    }


In [ ]:
# ========== chat：Gradio 回调 —— 历史转换 + 工具循环 ==========

def chat(message, history):
    # 将 Gradio messages 历史转换为 GenAI Content 列表
    contents = []

    for item in history:
        role = item["role"]

        # 跳过 system：Gemma 侧用 system_instruction，而不是 content 里的 system 角色
        if role == "system":
            continue

        # 将 OpenAI 风格 role 映射到 GenAI：assistant → model
        if role == "assistant":
            genai_role = "model"
        else:
            genai_role = "user"

        contents.append(
            types.Content(
                role=genai_role,
                parts=[types.Part.from_text(text=item["content"])]
            )
        )

    # 添加当前用户消息（本轮刚输入的那句）
    contents.append(
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=message)]
        )
    )

    # 示例：可按关键词动态追加系统说明（演示「动态 system」；实际 config 仍用 system_message）
    relevant_system_message = system_message
    if 'belt' in message.lower(): # example
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."

    # tools= 直接挂 Python 函数；SDK 负责声明与调度形态
    config = types.GenerateContentConfig(
        system_instruction=system_message,
        tools=[
            get_ticket_prices,
            set_price_ticket
            ]
    )

    # 第一次打电话给杰玛（首轮 generate）
    response = client.models.generate_content(
        model="gemma-4-31b-it",
        contents=contents,
        config=config,
    )

    # 本地可执行工具注册表：名字必须和模型 function_call.name 对得上
    AVAILABLE_FUNCTIONS = {
        "get_ticket_prices": get_ticket_prices,
        "set_price_ticket": set_price_ticket,
    }

    # 若有候选回复，检查其中是否包含 function_call
    if response.candidates:
        for part in response.candidates[0].content.parts:

            # 不是工具调用的 part 就跳过
            if not part.function_call:
                continue

            function_name = part.function_call.name

            # 按名字取出本地函数；未知工具则直接返回英文错误句
            function_to_call = AVAILABLE_FUNCTIONS.get(function_name)

            if function_to_call is None:
                return f"Unknown tool requested: {function_name}"

            # 把模型给的 args 解包成关键字参数并执行
            tool_result = function_to_call(
                    **dict(part.function_call.args)
                )

            # 先把「模型请求工具」的 content 追加进对话
            contents.append(response.candidates[0].content)

            # 再追加 role=tool 的 function_response
            contents.append(
                types.Content(
                    role="tool",
                    parts=[
                        types.Part.from_function_response(
                            name=function_name,
                            response=tool_result
                        )
                    ]
                )
            )

            # 带着工具结果再问一次模型，得到最终自然语言
            final_response = client.models.generate_content(
                model="gemma-4-31b-it",
                contents=contents,
                config=config,
            )

            return final_response.text

    # 没有工具调用：直接返回首轮文本
    return response.text


In [ ]:
# ========== 启动 Gradio ChatInterface ==========

# type="messages"：history 为 {role, content} 列表，与上面 chat() 解析方式一致
gr.ChatInterface(fn=chat, type="messages").launch()
